# Orders Bronze Ingestion

Production-style, file-by-file ingestion from ADLS Landing to a Delta Bronze path, coordinated by the shared file manifest.

**Safety rules**
- Never reprocess a `SUCCESS` file automatically.
- Normalize `quantity` to `BIGINT` and `sales_amount` to `DOUBLE`.
- Verify file hash, manifest ownership, and lease before writing.
- Refuse to append if the same `source_file` already exists in Bronze.
- Mark a manifest row `SUCCESS` only after persisted row-count verification.
- If a write may have succeeded but a later step failed, inspect/reconcile instead of blindly rerunning.

In [0]:
# CELL 1 — Imports and configuration
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime, timezone
import hashlib
import uuid

STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
LANDING_PATH = BASE_PATH + "landing/orders/"
BRONZE_PATH = BASE_PATH + "bronze/orders/"
MANIFEST_PATH = BASE_PATH + "metadata/file_manifest/"

DATASET_NAME = "orders"
SOURCE_SYSTEM = "orders_api"

BRONZE_COLUMNS = [
    "customer_id", "market", "order_id", "order_timestamp",
    "product_id", "quantity", "sales_amount", "updated_at",
    "ingestion_timestamp", "source_system", "source_file",
    "run_id", "batch_id"
]

print("Landing:", LANDING_PATH)
print("Bronze:", BRONZE_PATH)
print("Manifest:", MANIFEST_PATH)

Landing: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/
Bronze: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/orders/
Manifest: abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/metadata/file_manifest/


In [0]:
# CELL 2 — Inspect manifest and Bronze state (READ ONLY)
if not DeltaTable.isDeltaTable(spark, MANIFEST_PATH):
    raise RuntimeError(f"Manifest Delta table not found: {MANIFEST_PATH}")

manifest_df = spark.read.format("delta").load(MANIFEST_PATH)
required_manifest_columns = {
    "dataset", "file_name", "file_path", "batch_id", "status",
    "run_id", "content_hash", "lease_expires_at"
}
missing = required_manifest_columns - set(manifest_df.columns)
if missing:
    raise ValueError(f"Manifest missing columns: {sorted(missing)}")

display(
    manifest_df.filter(F.col("dataset") == DATASET_NAME)
    .select("file_name", "file_path", "status", "run_id",
            "content_hash", "lease_expires_at", "retry_count", "error_message")
    .orderBy("file_name")
)

if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
    existing_bronze_df = spark.read.format("delta").load(BRONZE_PATH)
    display(
        existing_bronze_df.groupBy("source_file")
        .agg(F.count("*").alias("bronze_rows"))
        .orderBy("source_file")
    )
else:
    print("Bronze Delta table does not exist yet.")

file_name,file_path,status,run_id,content_hash,lease_expires_at,retry_count,error_message
orders_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=07/orders_20261007.json,SUCCESS,fefa2f30-5d58-4043-8570-89b3e58146d0,19d8de3c2b51f519b3528fd93aa15766dec2a0ece904609880c606b20ee682af,null,0,null
orders_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=08/orders_20261008.json,PROCESSING,44b4e882-f5a2-42e9-883d-220de063b85a,4a9412127a3e037d1afd64c27dd32d934b91351d607bcb20c9d710edfef92d3d,2026-10-11T11:28:37.645Z,1,null


source_file,bronze_rows
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=07/orders_20261007.json,25000


In [0]:
# CELL 3 — Claim eligible orders files for this run
# Eligible: NEW, FAILED, or PROCESSING with an expired lease.
# SUCCESS files are never selected. Existing PROCESSING rows with live leases are not stolen.
run_id = str(uuid.uuid4())
now_utc = datetime.now(timezone.utc).replace(tzinfo=None)
lease_minutes = 60

manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)
current = spark.read.format("delta").load(MANIFEST_PATH).filter(
    F.col("dataset") == DATASET_NAME
)

eligible = current.filter(
    (F.col("status").isin("NEW", "FAILED")) |
    (
        (F.col("status") == "PROCESSING") &
        F.col("lease_expires_at").isNotNull() &
        (F.col("lease_expires_at") <= F.lit(now_utc))
    )
)

eligible_rows = eligible.select("file_path", "file_name", "status", "run_id").collect()
if not eligible_rows:
    print("No eligible orders files to claim. Check Cell 2 output.")
    claimed_paths = []
else:
    for r in eligible_rows:
        old_status = r["status"]
        old_run_id = r["run_id"]
        condition = (
            (F.col("dataset") == F.lit(DATASET_NAME)) &
            (F.col("file_path") == F.lit(r["file_path"])) &
            (F.col("status") == F.lit(old_status))
        )
        if old_run_id is None:
            condition = condition & F.col("run_id").isNull()
        else:
            condition = condition & (F.col("run_id") == F.lit(old_run_id))
        # Compare-and-set style claim. Only rows still matching their observed state are updated.
        manifest_delta.update(
            condition=condition,
            set={
                "status": F.lit("PROCESSING"),
                "run_id": F.lit(run_id),
                "claimed_at": F.current_timestamp(),
                "lease_expires_at": F.expr(f"current_timestamp() + INTERVAL {lease_minutes} MINUTES"),
                "last_attempt_at": F.current_timestamp(),
                "attempt_count": F.coalesce(F.col("attempt_count"), F.lit(0)) + F.lit(1),
                "status_updated_at": F.current_timestamp(),
                "error_message": F.lit(None).cast("string"),
                "error_type": F.lit(None).cast("string")
            }
        )

    claimed_paths = [
        r["file_path"] for r in
        spark.read.format("delta").load(MANIFEST_PATH)
        .filter(
            (F.col("dataset") == DATASET_NAME) &
            (F.col("status") == "PROCESSING") &
            (F.col("run_id") == run_id)
        ).select("file_path").collect()
    ]

print("Run ID:", run_id)
print("Claimed file count:", len(claimed_paths))
display(
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME) &
        (F.col("run_id") == run_id)
    )
    .select("file_name", "status", "run_id", "lease_expires_at")
    .orderBy("file_name")
)

No eligible orders files to claim. Check Cell 2 output.
Run ID: 83be1d01-6162-4b78-bc57-09e6b894e93e
Claimed file count: 0


file_name,status,run_id,lease_expires_at


In [0]:
# CELL 4 — Define manifest status updater
def update_manifest_status(
    file_path,
    new_status,
    source_record_count=None,
    written_record_count=None,
    rejected_record_count=None,
    processed_content_hash=None,
    error_message=None,
    error_type=None
):
    allowed = {"SUCCESS", "FAILED"}
    if new_status not in allowed:
        raise ValueError(f"Unsupported terminal status: {new_status}")

    set_values = {
        "status": F.lit(new_status),
        "processed_at": F.current_timestamp() if new_status == "SUCCESS" else F.col("processed_at"),
        "status_updated_at": F.current_timestamp(),
        "claimed_at": F.lit(None).cast("timestamp"),
        "lease_expires_at": F.lit(None).cast("timestamp"),
        "error_message": F.lit(error_message).cast("string") if error_message else F.lit(None).cast("string"),
        "error_type": F.lit(error_type).cast("string") if error_type else F.lit(None).cast("string")
    }
    if new_status == "SUCCESS":
        set_values.update({
            "last_success_at": F.current_timestamp(),
            "source_record_count": F.lit(source_record_count).cast("long"),
            "written_record_count": F.lit(written_record_count).cast("long"),
            "rejected_record_count": F.lit(rejected_record_count or 0).cast("long"),
            "processed_content_hash": F.lit(processed_content_hash)
        })
    else:
        set_values.update({
            "retry_count": F.coalesce(F.col("retry_count"), F.lit(0)) + F.lit(1),
            "last_error_at": F.current_timestamp()
        })

    dt = DeltaTable.forPath(spark, MANIFEST_PATH)
    dt.update(
        condition=(
            (F.col("dataset") == F.lit(DATASET_NAME)) &
            (F.col("file_path") == F.lit(file_path)) &
            (F.col("status") == F.lit("PROCESSING")) &
            (F.col("run_id") == F.lit(run_id))
        ),
        set=set_values
    )

    # Verify status update actually affected the row.
    verify = (
        spark.read.format("delta").load(MANIFEST_PATH)
        .filter(
            (F.col("dataset") == DATASET_NAME) &
            (F.col("file_path") == file_path)
        )
        .select("status", "run_id")
        .collect()
    )
    if len(verify) != 1 or verify[0]["status"] != new_status or verify[0]["run_id"] != run_id:
        raise RuntimeError(f"Manifest status update could not be verified for {file_path}")

In [0]:
# CELL 5 — Read, normalize, and validate claimed files (NO WRITES)
claimed_rows = (
    spark.read.format("delta").load(MANIFEST_PATH)
    .filter(
        (F.col("dataset") == DATASET_NAME) &
        (F.col("status") == "PROCESSING") &
        (F.col("run_id") == run_id)
    )
    .select("file_name", "file_path", "batch_id", "content_hash", "lease_expires_at")
    .orderBy("file_name")
    .collect()
)

prepared_by_path = {}
source_counts = {}
hash_by_path = {}

for r in claimed_rows:
    p = r["file_path"]
    if not r["content_hash"]:
        raise RuntimeError(f"Manifest content_hash missing for {r['file_name']}")
    if r["lease_expires_at"] is None:
        raise RuntimeError(f"Lease missing for {r['file_name']}")

    # Hash the exact file bytes from ADLS.
    file_bytes = spark.sparkContext.binaryFiles(p).first()[1].toArray()
    actual_hash = hashlib.sha256(file_bytes).hexdigest()
    if actual_hash != r["content_hash"]:
        raise RuntimeError(f"File hash mismatch for {r['file_name']}")

    source = spark.read.option("multiLine", "true").json(p)
    required = [
        "customer_id", "market", "order_id", "order_timestamp",
        "product_id", "quantity", "sales_amount", "updated_at"
    ]
    missing = [c for c in required if c not in source.columns]
    if missing:
        raise ValueError(f"{r['file_name']} missing columns: {missing}")

    normalized = (
        source
        .withColumn("customer_id", F.expr("try_cast(customer_id AS BIGINT)"))
        .withColumn("market", F.col("market").cast("string"))
        .withColumn("order_id", F.expr("try_cast(order_id AS BIGINT)"))
        .withColumn("order_timestamp", F.col("order_timestamp").cast("string"))
        .withColumn("product_id", F.expr("try_cast(product_id AS BIGINT)"))
        .withColumn("quantity", F.expr("try_cast(trim(cast(quantity AS STRING)) AS BIGINT)"))
        .withColumn("sales_amount", F.expr("try_cast(trim(cast(sales_amount AS STRING)) AS DOUBLE)"))
        .withColumn("updated_at", F.col("updated_at").cast("string"))
    )
    source_count = normalized.count()
    if source_count == 0:
        raise ValueError(f"{r['file_name']} is empty")

    invalid_exprs = [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in ["customer_id", "order_id", "product_id", "quantity", "sales_amount"]
    ]
    invalid = normalized.select(*invalid_exprs).first().asDict()
    invalid = {k: (v or 0) for k, v in invalid.items()}
    if any(v > 0 for v in invalid.values()):
        raise ValueError(f"{r['file_name']} has invalid required numeric values: {invalid}")

    prepared = (
        normalized
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("source_file", F.lit(p))
        .withColumn("run_id", F.lit(run_id))
        .withColumn("batch_id", F.lit(r["batch_id"]))
        .select(*BRONZE_COLUMNS)
    )

    prepared_by_path[p] = prepared
    source_counts[p] = source_count
    hash_by_path[p] = actual_hash
    print(f"Validated {r['file_name']}: {source_count} rows; invalid numeric values: 0")

print("Validated file count:", len(prepared_by_path))

Validated file count: 0


In [0]:
# CELL 6 — Controlled file-by-file Bronze write
# If a file already has rows in Bronze, refuse to append. Inspect/reconcile instead.
successful = []
failed = []

for r in claimed_rows:
    p = r["file_path"]
    name = r["file_name"]
    try:
        # Recheck current ownership and lease immediately before write.
        current = (
            spark.read.format("delta").load(MANIFEST_PATH)
            .filter(
                (F.col("dataset") == DATASET_NAME) &
                (F.col("file_path") == p) &
                (F.col("status") == "PROCESSING") &
                (F.col("run_id") == run_id)
            )
            .select("content_hash", "lease_expires_at")
            .collect()
        )
        if len(current) != 1:
            raise RuntimeError("Manifest ownership check failed")
        if current[0]["content_hash"] != hash_by_path[p]:
            raise RuntimeError("Manifest hash changed after validation")
        expiry = current[0]["lease_expires_at"]
        now = datetime.now(timezone.utc).replace(tzinfo=None)
        if expiry is None or expiry <= now:
            raise RuntimeError("Manifest lease is missing or expired")

        # Re-hash the file immediately before writing.
        file_bytes = spark.sparkContext.binaryFiles(p).first()[1].toArray()
        current_hash = hashlib.sha256(file_bytes).hexdigest()
        if current_hash != hash_by_path[p]:
            raise RuntimeError("Source file changed after validation")

        if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
            existing_count = (
                spark.read.format("delta").load(BRONZE_PATH)
                .filter(F.col("source_file") == p).count()
            )
            if existing_count != 0:
                raise RuntimeError(
                    f"Bronze already has {existing_count} rows for this file; refusing duplicate append"
                )

        expected_schema = [
            (f.name, f.dataType.simpleString())
            for f in prepared_by_path[p].schema
        ]
        # For an existing target, require exact ordered column/type compatibility.
        if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
            target_schema = [
                (f.name, f.dataType.simpleString())
                for f in spark.read.format("delta").load(BRONZE_PATH).schema
            ]
            if expected_schema != target_schema:
                raise RuntimeError(
                    f"Schema mismatch for {name}. Prepared={expected_schema}; target={target_schema}"
                )

        (prepared_by_path[p].write.format("delta").mode("append").save(BRONZE_PATH))

        persisted_count = (
            spark.read.format("delta").load(BRONZE_PATH)
            .filter(F.col("source_file") == p).count()
        )
        if persisted_count != source_counts[p]:
            raise RuntimeError(
                f"Persisted row count mismatch: expected {source_counts[p]}, found {persisted_count}. "
                "Inspect Bronze before retrying."
            )

        update_manifest_status(
            file_path=p,
            new_status="SUCCESS",
            source_record_count=source_counts[p],
            written_record_count=persisted_count,
            rejected_record_count=0,
            processed_content_hash=current_hash
        )
        successful.append(name)
        print(f"SUCCESS: {name} — {persisted_count} rows verified")

    except Exception as exc:
        msg = f"{type(exc).__name__}: {str(exc)}"[:4000]
        print(f"FAILED: {name}: {msg}")
        failed.append((name, msg))
        # Only mark failed if the row is still owned and PROCESSING.
        try:
            owned = (
                spark.read.format("delta").load(MANIFEST_PATH)
                .filter(
                    (F.col("dataset") == DATASET_NAME) &
                    (F.col("file_path") == p) &
                    (F.col("status") == "PROCESSING") &
                    (F.col("run_id") == run_id)
                ).count()
            )
            if owned == 1:
                # If rows exist, do not blindly change status: require reconciliation.
                bronze_has_rows = False
                if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
                    bronze_has_rows = (
                        spark.read.format("delta").load(BRONZE_PATH)
                        .filter(F.col("source_file") == p).limit(1).count() > 0
                    )
                if not bronze_has_rows:
                    update_manifest_status(
                        file_path=p,
                        new_status="FAILED",
                        error_message=msg,
                        error_type=type(exc).__name__
                    )
                else:
                    print("RECONCILIATION REQUIRED: Bronze rows exist; manifest left PROCESSING.")
        except Exception as status_exc:
            print("Could not safely update manifest status:", str(status_exc)[:1000])

print("Successful files:", successful)
print("Failed files:", failed)

Successful files: []
Failed files: []


In [0]:
# CELL 7 — Final read-only verification
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
manifest_final = spark.read.format("delta").load(MANIFEST_PATH)

display(
    bronze_df.filter(F.col("source_system") == SOURCE_SYSTEM)
    .groupBy("source_file")
    .agg(
        F.count("*").alias("bronze_rows"),
        F.countDistinct("order_id").alias("distinct_order_ids"),
        F.sum(F.when(F.col("order_id").isNull(), 1).otherwise(0)).alias("null_order_ids")
    )
    .orderBy("source_file")
)

display(
    manifest_final.filter(F.col("dataset") == DATASET_NAME)
    .select("file_name", "status", "run_id", "source_record_count",
            "written_record_count", "retry_count", "error_message")
    .orderBy("file_name")
)

print("Bronze total orders rows:", bronze_df.filter(F.col("source_system") == SOURCE_SYSTEM).count())

source_file,bronze_rows,distinct_order_ids,null_order_ids
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=07/orders_20261007.json,25000,25000,0


file_name,status,run_id,source_record_count,written_record_count,retry_count,error_message
orders_20261007.json,SUCCESS,fefa2f30-5d58-4043-8570-89b3e58146d0,25000,25000,0,null
orders_20261008.json,PROCESSING,44b4e882-f5a2-42e9-883d-220de063b85a,null,null,1,null


Bronze total orders rows: 25000
